In [5]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# ---------------------------------------------------------
# 1. LOAD DATA PREPROCESSED
# ---------------------------------------------------------
print("Memuat dataset...")
df = pd.read_csv('../data/processed/github_issues_preprocessed.csv')

# Deteksi kolom teks (karena hasil preprocessing biasanya bernama 'text', bukan 'body')
text_candidates = ['body', 'text', 'clean_text', 'title', 'description', 'content']
text_col = next((c for c in text_candidates if c in df.columns), None)
if text_col is None:
    raise KeyError(f"Tidak menemukan kolom teks. Kolom yang tersedia: {df.columns.tolist()}")
print(f"Kolom teks yang dipakai: {text_col}")

# Validasi label column
if 'severity' not in df.columns:
    raise KeyError(f"Tidak menemukan kolom label 'severity'. Kolom yang tersedia: {df.columns.tolist()}")

# Normalisasi label (aman walaupun sudah dinormalisasi)
df['severity'] = df['severity'].replace(['Major', 'Minor'], 'Non-Critical')

# Buang baris yang teks/label-nya kosong
df = df.dropna(subset=[text_col, 'severity']).reset_index(drop=True)

print("\n=== JUMLAH DATA YANG DIPAKAI ===")
print(f"Total data setelah dropna: {len(df)}")

Memuat dataset...
Kolom teks yang dipakai: text

=== JUMLAH DATA YANG DIPAKAI ===
Total data setelah dropna: 112264


In [6]:
# 2. LABEL ENCODING (Ubah Teks Kelas jadi Angka)
# ---------------------------------------------------------
le = LabelEncoder()
y = le.fit_transform(df['severity'])

print("\n=== INFORMASI KELAS ===")
print("Kelas yang terdeteksi oleh sistem:", le.classes_)

unique, counts = np.unique(y, return_counts=True)
for val, count in zip(unique, counts):
    print(f"Kelas {val} ({le.classes_[val]}): {count} data")


=== INFORMASI KELAS ===
Kelas yang terdeteksi oleh sistem: ['Critical' 'Non-Critical']
Kelas 0 (Critical): 65527 data
Kelas 1 (Non-Critical): 46737 data


In [8]:
# 3. TRAIN-TEST SPLIT (Bagi Data Latih & Uji)
# ---------------------------------------------------------
# stratify=y memastikan rasio kelas seimbang di Train dan Test
X_text = df['text']
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n=== PEMBAGIAN DATA ===")
print(f"Jumlah Data Train: {len(X_train_text)}")
print(f"Jumlah Data Test : {len(X_test_text)}")


=== PEMBAGIAN DATA ===
Jumlah Data Train: 89811
Jumlah Data Test : 22453


In [9]:
# 4. EKSTRAKSI FITUR: TF-IDF
# ---------------------------------------------------------
print("\nMelakukan proses TF-IDF (ini mungkin memakan waktu beberapa detik)...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,  
    ngram_range=(1, 3),  
    min_df=5,  
    max_df=0.8,  
    lowercase=True,
    stop_words=None 
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print(f"Bentuk (Shape) X_train_tfidf: {X_train_tfidf.shape}")
print(f"Bentuk (Shape) X_test_tfidf : {X_test_tfidf.shape}")


Melakukan proses TF-IDF (ini mungkin memakan waktu beberapa detik)...
Bentuk (Shape) X_train_tfidf: (89811, 10000)
Bentuk (Shape) X_test_tfidf : (22453, 10000)


In [10]:
# 5. SIMPAN KE FILE .PKL UNTUK MODELING & STREAMLIT
data_splits_tfidf = {
    'X_train': X_train_tfidf,
    'X_test': X_test_tfidf,
    'y_train': y_train,
    'y_test': y_test,
    'label_encoder': le
}

with open('../data/processed/data_splits_tfidf.pkl', 'wb') as f:
    pickle.dump(data_splits_tfidf, f)

# Menyimpan TF-IDF nya untuk dipakai di Streamlit nanti
with open('../data/processed/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("\nFile data_splits_tfidf.pkl dan tfidf_vectorizer.pkl berhasil diperbarui.")


File data_splits_tfidf.pkl dan tfidf_vectorizer.pkl berhasil diperbarui.
